In [0]:
dbutils.secrets.list(scope="tejscope")

In [0]:
dbutils.secrets.get(scope="tejscope", key="adlsaccountkey")

In [0]:
%python
# Set the configuration for accessing the Azure Data Lake Storage account
spark.conf.set(
    "fs.azure.account.key.adlsaccountfortejas.dfs.core.windows.net",
    dbutils.secrets.get(scope="tejscope", key="adlsaccountkey")
)

# Load the CSV file from Azure Data Lake Storage
df = spark.read.format("csv").option("header", "true").load("abfss://landing@adlsaccountfortejas.dfs.core.windows.net/")

display(df)

In [0]:
df = spark.read.option("multiline", "true").json(
  "abfss://landing@adlsaccountfortejas.dfs.core.windows.net/users.json"
)
df.show()

In [0]:
# Load the CSV file from Azure Data Lake Storage
df = spark.read.format("csv").option("header", "true").load("abfss://landing@adlsaccountfortejas.dfs.core.windows.net/")

display(df)

# PySpark Transformation

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df2=df.withColumn('AMOUNTINFOplus',col('AMOUNTINFO')+100)
df2.show()

In [0]:
df3=df.withColumn('NewORDERDATE',date_add(col('ORDERDATE'), 10))
df3.display()

In [0]:
dbutils.widgets.text("name", "Tej")
var = dbutils.widgets.get("name")
print(var)

In [0]:
df4=df.withColumn("flag",lit(var))
display(df4)

In [0]:
df5=df.withColumn("USERID",col("USERID").cast(IntegerType()))
display(df5)

###Create External Table

In [0]:
dbutils.secrets.list(scope="tejscope")

In [0]:
dbutils.secrets.get(scope="tejscope", key="adlsaccountkey")

In [0]:
%python
# Set the configuration for accessing the Azure Data Lake Storage account
spark.conf.set(
    "fs.azure.account.key.adlsaccountfortejas.dfs.core.windows.net",
    dbutils.secrets.get(scope="tejscope", key="adlsaccountkey")
)

In [0]:
%sql
use ProductDB;
create table ProductTableExt(
  id int,
  name string,
  price int
)
using delta
location 'abfss://landing@adlsaccountfortejas.dfs.core.windows.net/ProductDB/ExternalTable'

In [0]:
dbutils.secrets.get(scope="tejscope", key="adlsaccountkey")

In [0]:
'abfss://landing@adlsaccountfortejas.dfs.core.windows.net/ProductDB/ExternalTable';

In [0]:
%sql
use database Product;
create table ProductTableExt(
  id int,
  name string,
  price int
)
using delta
location 'abfss://landing@adlsaccountfortejas.dfs.core.windows.net/ProductDB/ExternalTable'

In [0]:
%sql
create database newDB;

create table ProductTableExt(
  id int,
  name string,
  price int
)
using delta
location 'abfss://landing@adlsaccountfortejas.dfs.core.windows.net/ProductDB/ExternalTable'

In [0]:
%sql
use database ProductDB;
insert into ProductTableExt(id,name,price) values(1,'Product1',100),(2,'Product2',200),(3,'Product3',300),(4,'Product4',400),(5,'Product5',500)

In [0]:
%sql
select * from ProductDB.ProductTableExt;

In [0]:
%sql
update ProductDB.ProductTableExt set price=2000 where id=3;
    
select * from ProductDB.ProductTableExt;

In [0]:
%sql 
delete from ProductDB.ProductTableExt where id=4;

In [0]:
%sql
select * from ProductDB.ProductTableExt;

In [0]:
%sql
drop table ProductDB.ProductTableExt;

In [0]:
%sql
create database ProdDB;
create table ProductTableExt(
  id int,
  name string,
  price int
)
using delta
location 'abfss://landing@adlsaccountfortejas.dfs.core.windows.net/ProductDB/newTable'

In [0]:
%sql
use database ProductDB;
select * from ProductTableExt;

In [0]:
%sql
use database ProductDB;
insert into ProductTableExt(id,name,price) values(1,'Product1',100),(2,'Product2',200),(3,'Product3',300),(4,'Product4',400),(5,'Product5',500)

In [0]:
%sql
delete from ProductDB.ProductTableExt where id=4;
    
select * from ProductDB.ProductTableExt;

##Data Versioning

In [0]:
%sql
describe history ProductDB.ProductTableExt;

In [0]:
%sql
RESTORE TABLE ProductDB.ProductTableExt TO VERSION AS OF 1;

In [0]:
%sql
select * from ProductDB.ProductTableExt;

#Delta Table Optimization

##Optimize

In [0]:
%sql
select * from ProductDB.ProductTableExt;

In [0]:
%sql
optimize ProductDB.ProductTableExt;
select * from ProductDB.ProductTableExt;

##ZORDER BY

In [0]:
%sql
optimize ProductDB.ProductTableExt ZORDER BY (id);

In [0]:
%sql
select * from ProductDB.ProductTableExt;

In [0]:
%sql
update ProductDB.ProductTableExt set price=8000 where id=5;
select * from ProductDB.ProductTableExt;

##AUTOLOADER

###Streaming Dataframe

In [0]:
df=spark.readStream.format('cloudFiles')\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", "abfss://autodestination@adlsaccountfortejas.dfs.core.windows.net/checkpoints")\
    .load('abfss://autosource@adlsaccountfortejas.dfs.core.windows.net')

In [0]:
df.writeStream.format('Delta')\
    .option("checkpointLocation", "abfss://autodestination@adlsaccountfortejas.dfs.core.windows.net/checkpoint")\
        .trigger(processingTime='5 seconds')\
            .start("abfss://autodestination@adlsaccountfortejas.dfs.core.windows.net/data1")